# Tool Search: 다른 접근 방법

**먼저 Tool Search with Embeddings cookbook을 읽기를 권장합니다.**

이 cookbook의 목표는 Claude에서 tool search(실제로는 "tool discovery")를 사용하는 몇 가지 다른 접근 방법을 보여 주는 것입니다. 두 유용한 기법을 보여 줍니다:

1. "Search" 없이 tool을 발견할 수 있습니다. 모든 tool 이름을 system prompt에 포함하고 decribe_tool_tool을 제공하여 Claude context에 tool 전체를 불러옵니다.
2. 아직 context에 불러오지 않은 tool은 요청의 `tools` 목록에 전달할 필요가 없습니다. 관리할 애플리케이션 복잡성이 조금 늘 수 있지만, 수천 개의 tool에 잠재적으로 접근하면서도 요청을 작게 유지할 수 있습니다.

사용자는 Claude context(및 Messages 요청)를 최대한 집중된 상태로 유지하도록 tool search를 설계할 폭넓은 유연성을 가집니다.

## 사전 요구 사항

이 guide를 따르기 전에 다음을 준비하세요:

**필요한 지식**
- Python 기초 - 함수, dictionary, 기본 자료 구조를 편하게 사용
- Claude tool use 기본 이해 - 먼저 [Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)를 읽기를 권장합니다.

**필요한 도구**
- Python 3.11 이상
- Anthropic API key([여기서 발급](https://docs.anthropic.com/claude/reference/getting-started-with-the-api))


In [ ]:
%pip install -q anthropic python-dotenv

In [ ]:
import json

import anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic()

print("✓ Client initialized")

## tool library 정의하기

간단한 tool 5개를 정의합니다. production에서는 database나 configuration 파일에 저장된 수백 또는 수천 개일 수 있습니다.


In [ ]:
TOOL_LIBRARY = {
    "get_weather": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
            },
            "required": ["city"],
        },
    },
    "get_stock_price": {
        "name": "get_stock_price",
        "description": "Get current stock price for a ticker symbol",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker (e.g., AAPL)"},
            },
            "required": ["ticker"],
        },
    },
    "convert_currency": {
        "name": "convert_currency",
        "description": "Convert amount between currencies",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    "calculate_tip": {
        "name": "calculate_tip",
        "description": "Calculate tip amount for a bill",
        "input_schema": {
            "type": "object",
            "properties": {
                "bill_amount": {"type": "number"},
                "tip_percent": {"type": "number", "default": 20},
            },
            "required": ["bill_amount"],
        },
    },
    "send_email": {
        "name": "send_email",
        "description": "Send an email to a recipient",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email"},
                "subject": {"type": "string"},
                "body": {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
}

print(f"✓ Defined {len(TOOL_LIBRARY)} tools: {list(TOOL_LIBRARY.keys())}")

## `describe_tool` tool

semantic search 대신 간단한 `describe_tool`을 제공합니다. Claude는 tool 이름으로 호출하여 해당 tool을 context에 불러옵니다.

system prompt가 사용 가능한 모든 tool 이름을 나열하므로 embedding이나 검색 없이 무엇이 있는지 압니다.


In [ ]:
DESCRIBE_TOOL = {
    "name": "describe_tool",
    "description": "Load a tool's full definition into context. Call this before using any tool for the first time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "tool_name": {
                "type": "string",
                "description": "Name of the tool to load",
            },
        },
        "required": ["tool_name"],
    },
}

# Tool catalog로 system prompt 구성
tool_names = list(TOOL_LIBRARY.keys())
SYSTEM_PROMPT = f"""You are a helpful assistant with access to various tools.

Available tools: {", ".join(tool_names)}

Before using any tool, you must first call describe_tool with the tool name to load it."""

print("System prompt:")
print(SYSTEM_PROMPT)

## 모의 tool 실행

시연용 간단한 모의 응답입니다:


In [ ]:
def execute_tool(name: str, inputs: dict) -> str:
    """Mock tool 실행."""
    if name == "get_weather":
        return json.dumps({"city": inputs["city"], "temp": "72°F", "conditions": "Sunny"})
    elif name == "get_stock_price":
        return json.dumps({"ticker": inputs["ticker"], "price": 185.50, "change": "+1.2%"})
    elif name == "convert_currency":
        rate = 0.92 if inputs["to_currency"] == "EUR" else 1.0
        converted = inputs["amount"] * rate
        return json.dumps({"converted": round(converted, 2), "to": inputs["to_currency"]})
    elif name == "calculate_tip":
        tip = inputs["bill_amount"] * (inputs.get("tip_percent", 20) / 100)
        return json.dumps({"tip": round(tip, 2), "total": round(inputs["bill_amount"] + tip, 2)})
    elif name == "send_email":
        return json.dumps({"status": "sent", "to": inputs["to"]})
    return json.dumps({"error": f"Unknown tool: {name}"})


print("✓ Mock execution ready")

## 동적 tool loading의 대화 루프

핵심 패턴은 다음과 같습니다:

1. tool 목록에서 **`describe_tool`만으로 시작**합니다.
2. **Claude가 `describe_tool`을 호출하면** `tool_reference`를 반환하고 `defer_loading=True`로 `active_tools`에도 tool을 추가합니다.
3. **`defer_loading=True`가 중요합니다** - tool 정의를 cache된 prompt prefix에서 제외하여 발견 시 cache invalidation을 막습니다.

대화에서 `tool_reference`를 볼 때마다 그 시점의 context에 전체 tool 정의가 불러와집니다.


In [ ]:
def run_conversation(user_message: str, max_turns: int = 10):
    """동적 tool loading으로 대화를 실행합니다."""
    print(f"\n{'=' * 60}")
    print(f"USER: {user_message}")
    print(f"{'=' * 60}\n")

    messages = [{"role": "user", "content": user_message}]

    # describe_tool만으로 시작 - 요청에 다른 tool 없음
    active_tools = [DESCRIBE_TOOL]
    loaded_tools = set()  # 추가한 tool 추적

    for turn in range(max_turns):
        print(f"--- Turn {turn + 1} (tools in request: {len(active_tools)}) ---")

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=active_tools,
            messages=messages,
            extra_headers={"anthropic-beta": "advanced-tool-use-2025-11-20"},
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if block.type == "text":
                    print(f"\nASSISTANT: {block.text}")
            break

        # Tool 호출 처리
        tool_results = []
        for block in response.content:
            if block.type == "text" and block.text:
                print(f"ASSISTANT: {block.text}")

            elif block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input

                if tool_name == "describe_tool":
                    requested_tool = tool_input["tool_name"]
                    print(f"🔍 describe_tool({requested_tool})")

                    if requested_tool in TOOL_LIBRARY:
                        # defer_loading=True로 active_tools에 tool 추가
                        # 이는 prompt caching에 매우 중요합니다!
                        if requested_tool not in loaded_tools:
                            tool_def = {**TOOL_LIBRARY[requested_tool], "defer_loading": True}
                            active_tools.append(tool_def)
                            loaded_tools.add(requested_tool)
                            print(f"   ✓ Added {requested_tool} to tools (defer_loading=True)")

                        # Claude가 사용할 수 있도록 tool_reference 반환
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": [
                                    {"type": "tool_reference", "tool_name": requested_tool}
                                ],
                            }
                        )
                    else:
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": f"Tool '{requested_tool}' not found.",
                            }
                        )
                else:
                    # 발견한 tool 실행
                    print(f"🔧 {tool_name}({json.dumps(tool_input)})")
                    result = execute_tool(tool_name, tool_input)
                    print(f"   → {result}")
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )

        if tool_results:
            messages.append({"role": "user", "content": tool_results})

    print(f"\n{'=' * 60}\n")


print("✓ Conversation loop ready")

## 예제: 날씨 query

다음 동작을 살펴보세요:
1. system prompt의 tool 목록에서 `get_weather`를 봅니다.
2. 불러오기 위해 `describe_tool("get_weather")`를 호출합니다.
3. `tool_reference`를 받고 이제 tool을 사용할 수 있습니다.


In [ ]:
run_conversation("What's the weather in Tokyo?")

## 예제: 여러 tool query

필요에 따라 여러 tool을 불러올 수 있습니다:


In [ ]:
run_conversation("Convert $100 to EUR, then calculate a 20% tip on a $85 dinner bill.")

## `defer_loading=True`가 중요한 이유

`tools` 목록에 tool을 추가하면 보통 정의가 context window 맨 앞에 불러와집니다. 새 tool을 추가하면 맨 앞이 바뀌어 cache 대부분을 잃습니다.

**`defer_loading=True`에서는:**
- tool 정의가 context window의 시작에 포함되지 않습니다.
- 대신 `tool_reference`를 볼 때 불러와집니다.
- 따라서 새 tool을 발견해도 system prompt와 초기 tool은 cache 상태를 유지합니다.

**패턴:**
```python
# 초기 요청 - describe_tool만 있으며 system prompt는 cache됨
tools = [DESCRIBE_TOOL]

# Claude가 describe_tool("get_weather")를 호출한 뒤
# cache 보존을 위해 defer_loading으로 추가
tools.append({**TOOL_LIBRARY["get_weather"], "defer_loading": True})

# 사용 가능함을 알도록 tool_reference 반환
tool_result = [{"type": "tool_reference", "tool_name": "get_weather"}]
```

수백 또는 수천 tool의 애플리케이션에서 다음을 원한다면 필수입니다:
- 초기 요청 크기 작게 유지
- tool 발견 사이 prompt caching 보존
- 실제 필요한 tool만 불러오기


## 결론

핵심 통찰은 **Claude가 필요로 하기 전까지 tool은 `tools` 목록에 없어도 된다**는 것입니다. `defer_loading=True` 및 `tool_reference`와 함께 요청을 작게 유지하고 prompt caching을 보존하면서 수천 tool로 확장할 수 있습니다. 다만 이 경우 client가 tool discovery 방식을 제공해야 합니다.

여기의 `describe_tool` 방식은 하나의 형태일 뿐입니다. 다른 패턴은 다음과 같습니다:
- **`list_tools`** - category나 keyword에 맞는 tool 이름 반환
- **계층적인 발견** - category를 탐색한 뒤 특정 tool 불러오기
- **Hybrid** - 큰 catalog에서 목록과 semantic search 결합

핵심 패턴은 항상 같습니다:
1. 발견 시 `tool_reference` 반환
2. caching 보존을 위해 `defer_loading=True`로 추가
3. 즉시 tool 사용 가능

embedding 기반 방식은 [Tool Search with Embeddings](./tool_search_with_embeddings.ipynb)를 참고하세요.
